- _exponent._bronze_allscripts_tw_works.dbo_encounter_other
- _exponent._bronze_allscripts_tw_works.dbo_encounter_itemchild
- _exponent._bronze_allscripts_tw_works.dbo_site_de

### Source Tables:
- _exponent._bronze_allscripts_tw_works.dbo_order_activity_header
- _exponent._bronze_allscripts_tw_works.dbo_encounter_itemchild (for visit details if needed)

### To Do:
- Map visit_concept_id (visit type - inpatient, outpatient, ER, etc.)
- Map visit_type_concept_id (EHR record vs Claims record, etc.)
- Add visit start/end dates from encounter details
- Link to care_site_id and provider_id once those tables are populated
- Add merge logic to gold to look for changes and not blanket overwrite

### Notes:
- CARE_SITE, PROVIDER, and PERSON must run before VISIT_OCCURRENCE
- Using dbo_order_activity_header to get unique EncounterID-PatientID pairs
- EncounterID is the visit/encounter identifier in Allscripts
- Currently using placeholder concept IDs - will need proper mapping

In [ ]:
%sql
-- Check how many unique encounters we have in the source
SELECT 
    COUNT(DISTINCT EncounterID) as unique_encounters,
    COUNT(DISTINCT PatientID) as unique_patients
FROM _exponent._bronze_allscripts_tw_works.dbo_order_activity_header
WHERE EncounterID IS NOT NULL 
  AND PatientID IS NOT NULL

# Transformation

In [ ]:
source = 'allscripts_tw'

In [ ]:
silver_visit_df = spark.sql(f'''
WITH unique_encounters AS (
  SELECT 
    EncounterID,
    PatientID,
    MIN(CreateDTTM) as first_activity_datetime,
    MAX(CreateDTTM) as last_activity_datetime
  FROM _exponent._bronze_allscripts_tw_works.dbo_order_activity_header
  WHERE EncounterID IS NOT NULL 
    AND PatientID IS NOT NULL
  GROUP BY EncounterID, PatientID
)
SELECT 
  -- visit_occurrence_id will be generated in source_to_visitor_occurrence
  source_to_person.person_id,
  9202 AS visit_concept_id,  -- TODO: Map to proper visit type (9202 = Outpatient Visit)
  CAST(e.first_activity_datetime AS DATE) AS visit_start_date,
  e.first_activity_datetime AS visit_start_datetime,
  CAST(COALESCE(e.last_activity_datetime, e.first_activity_datetime) AS DATE) AS visit_end_date,
  COALESCE(e.last_activity_datetime, e.first_activity_datetime) AS visit_end_datetime,
  32817 AS visit_type_concept_id,  -- 32817 = EHR
  NULL AS provider_id,  -- TODO: Link to provider once available
  NULL AS care_site_id,  -- TODO: Link to care_site once available
  CONCAT('{source}', ' | ', e.EncounterID) AS visit_source_value,
  0 AS visit_source_concept_id,
  0 AS admitted_from_concept_id,
  NULL AS admitted_from_source_value,
  0 AS discharged_to_concept_id,
  NULL AS discharged_to_source_value,
  NULL AS preceding_visit_occurrence_id,  -- TODO: Add logic for linking sequential visits
  '{source}' AS source_system
FROM unique_encounters e
INNER JOIN _exponent.omop_mapping.source_to_person
  ON CONCAT('{source}', ' | ', e.PatientID) = source_to_person.person_source_value
  AND source_to_person.active_flag = TRUE
''')

display(silver_visit_df)
silver_visit_df.createOrReplaceTempView("silver_visit_occurrence")

In [ ]:
%sql
MERGE INTO _exponent.omop_silver.visit_occurrence AS t
USING silver_visit_occurrence AS s
ON t.visit_source_value = s.visit_source_value

WHEN MATCHED AND (
     NOT (t.person_id <=> s.person_id)
  OR NOT (t.visit_concept_id <=> s.visit_concept_id)
  OR NOT (t.visit_start_date <=> s.visit_start_date)
  OR NOT (t.visit_start_datetime <=> s.visit_start_datetime)
  OR NOT (t.visit_end_date <=> s.visit_end_date)
  OR NOT (t.visit_end_datetime <=> s.visit_end_datetime)
  OR NOT (t.visit_type_concept_id <=> s.visit_type_concept_id)
  OR NOT (t.provider_id <=> s.provider_id)
  OR NOT (t.care_site_id <=> s.care_site_id)
  OR NOT (t.visit_source_concept_id <=> s.visit_source_concept_id)
  OR NOT (t.admitted_from_concept_id <=> s.admitted_from_concept_id)
  OR NOT (t.admitted_from_source_value <=> s.admitted_from_source_value)
  OR NOT (t.discharged_to_concept_id <=> s.discharged_to_concept_id)
  OR NOT (t.discharged_to_source_value <=> s.discharged_to_source_value)
  OR NOT (t.preceding_visit_occurrence_id <=> s.preceding_visit_occurrence_id)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.person_id                      = s.person_id,
  t.visit_concept_id               = s.visit_concept_id,
  t.visit_start_date               = s.visit_start_date,
  t.visit_start_datetime           = s.visit_start_datetime,
  t.visit_end_date                 = s.visit_end_date,
  t.visit_end_datetime             = s.visit_end_datetime,
  t.visit_type_concept_id          = s.visit_type_concept_id,
  t.provider_id                    = s.provider_id,
  t.care_site_id                   = s.care_site_id,
  t.visit_source_concept_id        = s.visit_source_concept_id,
  t.admitted_from_concept_id       = s.admitted_from_concept_id,
  t.admitted_from_source_value     = s.admitted_from_source_value,
  t.discharged_to_concept_id       = s.discharged_to_concept_id,
  t.discharged_to_source_value     = s.discharged_to_source_value,
  t.preceding_visit_occurrence_id  = s.preceding_visit_occurrence_id,
  t.source_system                  = s.source_system

WHEN NOT MATCHED THEN INSERT (
  person_id,
  visit_concept_id,
  visit_start_date,
  visit_start_datetime,
  visit_end_date,
  visit_end_datetime,
  visit_type_concept_id,
  provider_id,
  care_site_id,
  visit_source_value,
  visit_source_concept_id,
  admitted_from_concept_id,
  admitted_from_source_value,
  discharged_to_concept_id,
  discharged_to_source_value,
  preceding_visit_occurrence_id,
  source_system
)
VALUES (
  s.person_id,
  s.visit_concept_id,
  s.visit_start_date,
  s.visit_start_datetime,
  s.visit_end_date,
  s.visit_end_datetime,
  s.visit_type_concept_id,
  s.provider_id,
  s.care_site_id,
  s.visit_source_value,
  s.visit_source_concept_id,
  s.admitted_from_concept_id,
  s.admitted_from_source_value,
  s.discharged_to_concept_id,
  s.discharged_to_source_value,
  s.preceding_visit_occurrence_id,
  s.source_system
);

In [ ]:
%sql
SELECT * FROM _exponent.omop_silver.visit_occurrence
LIMIT 10

In [ ]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_visitor_occurrence (
    source_system,
    visit_occurrence_source_value,
    person_id,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.visit_source_value AS visit_occurrence_source_value,
    s.person_id,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    current_timestamp() AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT 
        source_system, 
        visit_source_value,
        person_id
    FROM _exponent.omop_silver.visit_occurrence
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_visitor_occurrence x
  ON s.visit_source_value = x.visit_occurrence_source_value;

In [ ]:
%sql
SELECT * FROM _exponent.omop_mapping.source_to_visitor_occurrence
LIMIT 20

In [ ]:
%sql
MERGE INTO _exponent.omop.visit_occurrence AS gold_visit
USING (
  SELECT
    source_to_visitor_occurrence.visit_occurrence_id,
    s.person_id,
    s.visit_concept_id,
    s.visit_start_date,
    s.visit_start_datetime,
    s.visit_end_date,
    s.visit_end_datetime,
    s.visit_type_concept_id,
    s.provider_id,
    s.care_site_id,
    s.visit_source_value,
    s.visit_source_concept_id,
    s.admitted_from_concept_id,
    s.admitted_from_source_value,
    s.discharged_to_concept_id,
    s.discharged_to_source_value,
    s.preceding_visit_occurrence_id
  FROM _exponent.omop_silver.visit_occurrence s
  JOIN _exponent.omop_mapping.source_to_visitor_occurrence
    ON source_to_visitor_occurrence.visit_occurrence_source_value = s.visit_source_value
   AND source_to_visitor_occurrence.active_flag = TRUE
) AS src
ON gold_visit.visit_occurrence_id = src.visit_occurrence_id

WHEN MATCHED THEN UPDATE SET
  gold_visit.person_id                     = src.person_id,
  gold_visit.visit_concept_id              = src.visit_concept_id,
  gold_visit.visit_start_date              = src.visit_start_date,
  gold_visit.visit_start_datetime          = src.visit_start_datetime,
  gold_visit.visit_end_date                = src.visit_end_date,
  gold_visit.visit_end_datetime            = src.visit_end_datetime,
  gold_visit.visit_type_concept_id         = src.visit_type_concept_id,
  gold_visit.provider_id                   = src.provider_id,
  gold_visit.care_site_id                  = src.care_site_id,
  gold_visit.visit_source_value            = src.visit_source_value,
  gold_visit.visit_source_concept_id       = src.visit_source_concept_id,
  gold_visit.admitted_from_concept_id      = src.admitted_from_concept_id,
  gold_visit.admitted_from_source_value    = src.admitted_from_source_value,
  gold_visit.discharged_to_concept_id      = src.discharged_to_concept_id,
  gold_visit.discharged_to_source_value    = src.discharged_to_source_value,
  gold_visit.preceding_visit_occurrence_id = src.preceding_visit_occurrence_id

WHEN NOT MATCHED THEN INSERT (
  visit_occurrence_id,
  person_id,
  visit_concept_id,
  visit_start_date,
  visit_start_datetime,
  visit_end_date,
  visit_end_datetime,
  visit_type_concept_id,
  provider_id,
  care_site_id,
  visit_source_value,
  visit_source_concept_id,
  admitted_from_concept_id,
  admitted_from_source_value,
  discharged_to_concept_id,
  discharged_to_source_value,
  preceding_visit_occurrence_id
)
VALUES (
  src.visit_occurrence_id,
  src.person_id,
  src.visit_concept_id,
  src.visit_start_date,
  src.visit_start_datetime,
  src.visit_end_date,
  src.visit_end_datetime,
  src.visit_type_concept_id,
  src.provider_id,
  src.care_site_id,
  src.visit_source_value,
  src.visit_source_concept_id,
  src.admitted_from_concept_id,
  src.admitted_from_source_value,
  src.discharged_to_concept_id,
  src.discharged_to_source_value,
  src.preceding_visit_occurrence_id
);

In [ ]:
%sql
SELECT * FROM _exponent.omop.visit_occurrence
LIMIT 20